# Multimodal Hybrid Product Search on AlloyDB - With Data Prep

This notebook provides a step-by-step example of implementing Hybrid Search in [AlloyDB for PostgreSQL](https://cloud.google.com/products/alloydb?e=48754805&hl=en) for Cymbal Shops, a fictional retailer with a large eCommerce presence. It combines multimodal vector embeddings ([`multimodalembedding@001`](https://cloud.google.com/vertex-ai/generative-ai/docs/embeddings/get-multimodal-embeddings)), fulltext search ([Generalized Inverted Index](https://www.postgresql.org/docs/current/gin.html)), and [BM25](https://en.wikipedia.org/wiki/Okapi_BM25) sparse embeddings ([pgvector 0.7.0+](https://github.com/pgvector/pgvector?tab=readme-ov-file#sparse-vectors)) with [Reciprocal Rank Fusion](https://medium.com/@devalshah1619/mathematical-intuition-behind-reciprocal-rank-fusion-rrf-explained-in-2-mins-002df0cc5e2a) re-ranking for enhanced product search.


The high-level flow is as follows:
- Imports a sample retail dataset (based on [theLook eCommerce dataset](https://console.cloud.google.com/marketplace/product/bigquery-public-data/thelook-ecommerce)) into an AlloyDB cluster.
- Asynchronously generates product descriptions for 29,120 products using the Gemini 2.0 Flash model.
- Asynchronously generates product images for 29,120 products using the Imagen 3 model.
- Asynchronously generates multimodal embeddings for the product images and product descriptions using the `multimodalembedding@001` model.
- Generates sparse embeddings for products using BM25.
- TO DO: Creates a [GIN index](https://www.postgresql.org/docs/current/gin.html) for fast full-text search.
- TO DO: Creates [ScaNN vector indexes](https://cloud.google.com/blog/products/databases/understanding-the-scann-index-in-alloydb) for fast vector embedding queries. 
- TO DO: Demonstrates performing hybrid search (vector + fts + BM25)

## Basic Setup

You will need an AlloyDB for PostgreSQL instance to use this notebook. Create one now if you have not already created it.

### Define Variables

In [ ]:
# Update these variables to match your environment
project_id = "your-project"  # @param {type:"string"}
region = "your-region"  # @param {type:"string"}
vpc = "your-vpc"  # @param {type:"string"}
image_bucket = "your-bucket"  # @param {type:"string"}
index_bucket = "your-bucket"  # @param {type:"string"}
alloydb_cluster = "your-alloydb-cluster"  # @param {type:"string"}
alloydb_instance = "your-alloydb-instance"  # @param {type:"string"}
alloydb_password = input("Please provide a password to be used for 'postgres' database user: ")

# Don't change values below this line.
alloydb_database = "ecom" 
database_backup_uri = "gs://pr-public-demo-data/alloydb-retail-demo/data/ecom.sql"  # @param {type:"string"}


### Install Dependencies

In [ ]:
! pip install --quiet google-cloud-storage \
                      google-cloud-aiplatform \
                      pymilvus.model

### Connect Your Google Cloud Project

In [ ]:
# Configure gcloud.
!gcloud config set project {project_id}

### Configure Logging

In [ ]:
import logging
import sys

# Configure the root logger to output messages with INFO level or above
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(asctime)s[%(levelname)5s][%(name)14s] - %(message)s',  datefmt='%H:%M:%S', force=True)

### Enable APIs for AlloyDB and Vertex AI

You will need to enable these APIs in order to create an AlloyDB database and utilize Vertex AI as an embeddings service!

In [ ]:
!gcloud services enable alloydb.googleapis.com aiplatform.googleapis.com

### Initialize GenAI Client

In [ ]:
from google import genai
from google.genai import types

genai_client = genai.Client(
    vertexai=True, project=project_id, location=region
)

### Connect to the AlloyDB Cluster

This function will create a connection pool to your AlloyDB instance using the AlloyDB Python connector. The AlloyDB Python connector will automatically create secure connections to your AlloyDB instance using mTLS.

In [ ]:
! pip install asyncpg google.cloud.alloydb.connector

In [ ]:
import asyncpg

import sqlalchemy
from sqlalchemy.ext.asyncio import AsyncEngine, create_async_engine

from google.cloud.alloydb.connector import AsyncConnector, IPTypes

async def init_connection_pool(connector: AsyncConnector, db_name: str = alloydb_database, pool_size: int = 5) -> AsyncEngine:
    # initialize Connector object for connections to AlloyDB
    connection_string = f"projects/{project_id}/locations/{region}/clusters/{alloydb_cluster}/instances/{alloydb_instance}"

    async def getconn() -> asyncpg.Connection:
        conn: asyncpg.Connection = await connector.connect(
            connection_string,
            "asyncpg",
            user="postgres",
            password=alloydb_password,
            db=db_name,
            ip_type=IPTypes.PRIVATE,
        )
        return conn

    pool = create_async_engine(
        "postgresql+asyncpg://",
        async_creator=getconn,
        pool_size=pool_size,
        max_overflow=0,
        isolation_level='AUTOCOMMIT'
    )
    return pool

connector = AsyncConnector()

postgres_db_pool = await init_connection_pool(connector, "postgres")
ecom_db_pool = await init_connection_pool(connector, f"{alloydb_database}")

## Define Helper Functions

#### rest_api_helper()

In [ ]:
import requests
import google.auth
import json

# Get an access token based upon the current user
creds, _ = google.auth.default()
authed_session = google.auth.transport.requests.AuthorizedSession(creds)
access_token=creds.token

if project_id:
  authed_session.headers.update({"x-goog-user-project": project_id}) # Required to workaround a project quota bug

def rest_api_helper(
    session: requests.Session,
    url: str,
    http_verb: str,
    request_body: dict = None,
    params: dict = None
  ) -> dict:
  """Calls a REST API using a pre-authenticated requests Session."""

  headers = {"Content-Type": "application/json"}

  try:

    if http_verb == "GET":
      response = session.get(url, headers=headers, params=params)
    elif http_verb == "POST":
      response = session.post(url, json=request_body, headers=headers, params=params)
    elif http_verb == "PUT":
      response = session.put(url, json=request_body, headers=headers, params=params)
    elif http_verb == "PATCH":
      response = session.patch(url, json=request_body, headers=headers, params=params)
    elif http_verb == "DELETE":
      response = session.delete(url, headers=headers, params=params)
    else:
      raise ValueError(f"Unknown HTTP verb: {http_verb}")

    # Raise an exception for bad status codes (4xx or 5xx)
    response.raise_for_status()

    # Check if response has content before trying to parse JSON
    if response.content:
        return response.json()
    else:
        return {} # Return empty dict for empty responses (like 204 No Content)

  except requests.exceptions.RequestException as e:
      # Catch potential requests library errors (network, timeout, etc.)
      # Log detailed error information
      print(f"Request failed: {e}")
      if e.response is not None:
          print(f"Request URL: {e.request.url}")
          print(f"Request Headers: {e.request.headers}")
          print(f"Request Body: {e.request.body}")
          print(f"Response Status: {e.response.status_code}")
          print(f"Response Text: {e.response.text}")
          # Re-raise a more specific error or a custom one
          raise RuntimeError(f"API call failed with status {e.response.status_code}: {e.response.text}") from e
      else:
          raise RuntimeError(f"API call failed: {e}") from e
  except json.JSONDecodeError as e:
      print(f"Failed to decode JSON response: {e}")
      print(f"Response Text: {response.text}")
      raise RuntimeError(f"Invalid JSON received from API: {response.text}") from e



#### run_query()

In [ ]:
# Create AlloyDB Query Helper Function
from sqlalchemy import text, exc
import pandas as pd

async def run_query(pool, sql, output_as_df = True):
  """Executes a SQL query against the AlloyDB database.

  This function accepts a SQL string and performs the following actions:
  - If the SQL statement starts with 'SELECT' or 'WITH' (case-insensitive),
    it executes the query and returns the results as a Pandas DataFrame
    with column names derived from the query.
  - For other types of SQL statements (e.g., INSERT, UPDATE, DELETE),
    it executes the query and returns the SQLAlchemy ResultProxy object
    after committing the transaction.

  Args:
    sql: A string containing the SQL query to execute.

  Returns:
    pandas.DataFrame: If the SQL statement is a SELECT or WITH query,
      a DataFrame containing the query results.
    sqlalchemy.engine.result.ResultProxy: If the SQL statement is not a
      SELECT or WITH query, a ResultProxy object representing the
      result of the execution.
    None: If a `sqlalchemy.exc.ProgrammingError` occurs during query execution,
      the error is printed to the console, and None is returned.

  Raises:
    sqlalchemy.exc.ProgrammingError: If there is an issue with the SQL syntax
      or the database operation. The error is caught, printed, and None is
      returned.

  Example Usage:
  >>> # SELECT query
  >>> sql_select = "SELECT ticker, company_name from investments LIMIT 5"
  >>> df_result = await run_query(sql_select)
  >>> print(df_result)
  >>>
  >>> # INSERT query
  >>> sql_insert = "INSERT INTO investments (ticker, company_name) VALUES ('NEW', 'New Company')"
  >>> insert_result = await run_query(sql_insert)
  >>> print(insert_result) # Output will be the ResultProxy object
  """
  async with pool.connect() as conn:
    if sql.strip().lower().startswith('select') or sql.strip().lower().startswith('with'):
      try:
        result = await conn.execute(sqlalchemy.text(sql))
        if output_as_df:
          rows = result.fetchall()
          column_names = result.keys()
          df = pd.DataFrame(rows, columns=column_names)
          return df
        else:
          return result
      except exc.ProgrammingError as e:
        print(e)
    else:
      try:
        result = await conn.execute(
            text(sql)
        )
        await conn.commit()
        operation_type = sql.split()[0].upper()
        row_count = result.rowcount
        if operation_type in ['INSERT', 'UPDATE', 'DELETE']:
          print(f"{operation_type} statement executed successfully. {row_count} row(s) affected.")
        else:
          print(f"{operation_type} statement executed successfully.")
        return result

      except exc.ProgrammingError as e:
        print(e)



### retry_condition()

In [ ]:
from tenacity import retry, wait_exponential, stop_after_attempt, before_sleep_log, retry_if_exception, wait_fixed

def retry_condition(error):
  error_string = str(error)
  print(error_string)

  retry_errors = [
      "429 Quota exceeded",
      #"The prompt could not be submitted",
  ]

  for retry_error in retry_errors:
    if retry_error in error_string:
      print("Retrying...")
      return True

  return False

## Import Sample Data to AlloyDB

### Add Required Permissions

These permissions are required to read from GCS for the data import and to integrate with Vertex AI for on-the-fly embedding generation.

In [ ]:
project_number = ! gcloud projects describe {project_id} --format='value(projectNumber)'
project_number = project_number[0]

roles_array = [
    "roles/storage.admin",
    "roles/aiplatform.user"
]

for r in roles_array:
  ! gcloud projects add-iam-policy-binding {project_id} \
      --member="serviceAccount:service-{project_number}@gcp-sa-alloydb.iam.gserviceaccount.com" \
      --role="{r}"

### OPTIONAL: Drop Existing Database

In [ ]:
# Close existing connections to the database
sql = f"""SELECT pg_terminate_backend(pg_stat_activity.pid)
FROM pg_stat_activity
WHERE pg_stat_activity.datname = '{alloydb_database}'
  AND pid <> pg_backend_pid();"""
await run_query(postgres_db_pool, sql)

# Uncomment the lines below to drop an existing database before re-creating it
#sql = f"DROP DATABASE {alloydb_database};"
#await run_query(postgres_db_pool, sql)

# Reinitiate the connection pool
ecom_db_pool = await init_connection_pool(connector, f"{alloydb_database}")

### Create Database

In [ ]:
# Create the database
sql = f"CREATE DATABASE {alloydb_database};"
await run_query(postgres_db_pool, sql)

### Install Pre-requisite Extensions

In [ ]:
sql_array = []

sql_array.append("CREATE EXTENSION IF NOT EXISTS vector;")

sql_array.append("CREATE EXTENSION IF NOT EXISTS google_ml_integration;")

for sql in sql_array:
  await run_query(ecom_db_pool, sql)

### Run the Import

In [ ]:
# Reference: https://cloud.google.com/alloydb/docs/reference/rest/v1/projects.locations.clusters/import
#            https://cloud.google.com/alloydb/docs/import-sql-file

import time

url = f"https://alloydb.googleapis.com/v1/projects/{project_id}/locations/{region}/clusters/{alloydb_cluster}:import"
request_body = {
   "gcsUri": f"{database_backup_uri}",
   "database": f"{alloydb_database}",
   "user": "postgres",
   "sqlImportOptions": {}
}

result = rest_api_helper(authed_session, url, 'POST', request_body, {})
print(f"Kicked off export: {result}")

operation_id = result['name']

operation_complete = False
while operation_complete == False:
  print(f"Export still running: {operation_id}")
  url = f"https://alloydb.googleapis.com/v1/{operation_id}"
  response = rest_api_helper(authed_session, url, 'GET', request_body, {})
  operation_complete = response['done']
  if operation_complete:
    print(f"Operation complete. Check result payload for potential errors. \nResult: {response}")
    continue
  time.sleep(5)

### Check Row Counts

In [ ]:
sql = """
SELECT 'distribution_centers' AS table_name, (SELECT COUNT(*) FROM distribution_centers) AS actual_row_count, 10 AS target_row_count
UNION ALL
SELECT 'events', (SELECT COUNT(*) FROM events), 2438862
UNION ALL
SELECT 'inventory_items', (SELECT COUNT(*) FROM inventory_items), 494254
UNION ALL
SELECT 'orders', (SELECT COUNT(*) FROM orders), 125905
UNION ALL
SELECT 'order_items', (SELECT COUNT(*) FROM order_items), 182905
UNION ALL
SELECT 'products', (SELECT COUNT(*) FROM products), 29120
UNION ALL
SELECT 'users', (SELECT COUNT(*) FROM users), 100000;
"""

await run_query(ecom_db_pool, sql)

### Test the Vertex AI Integration

In [ ]:
sql = "SELECT embedding('text-embedding-005', 'This string will be transformed into an embedding.');"
await run_query(ecom_db_pool, sql)

## Generate Product Descriptions with Gemini 2.0 Flash

### Add product_description Column

In [ ]:
sql = """ALTER TABLE products ADD COLUMN product_description TEXT;"""
await run_query(ecom_db_pool, sql)

### build_product_description_prompt()

In [ ]:
def build_product_description_prompt(name, brand, category, department, retail_price, sku):
  prompt = f"""**Persona & Context:**
You are a skilled Copywriter for Cymbal Shops. Cymbal Shops is a specialty big box retailer offering a diverse, curated mix of trendy clothing, unique household knick-knacks, stylish furnishings, and interesting personal items. Our customers appreciate value, style, and finding items with personality. Our brand voice is:
* Approachable & Friendly
* Professional & Trustworthy
* Slightly Quirky & Stylish
* Focused on Value & Benefits

**Task:**
Write a compelling product description for the Cymbal Shops online product catalog page.

**Output Requirements:**
* **Length:** 50-75 words (approx. 3-5 sentences).
* **Tone:** Match the Cymbal Shops brand voice described above.
* **Goal:** Engage the customer and highlight the key benefits and appeal of the product. Translate features into benefits.
* **Format:** A single paragraph of prose. Optionally, include 2-3 key feature bullet points after the main paragraph if features are distinct and numerous.
* **Exclusions:** Do NOT mention the SKU or Retail Price within the written description.

**Product Information to Use:**
Product Name: {name}
Brand: {brand}
Category: {category}
Department: {department}
Retail Price: {retail_price}  (For context only, do not include in description)
SKU: {sku} (For context only, do not include in description)

**Now, write the description for the product detailed above.**
"""
  return prompt

### generate_text()

In [ ]:
async def generate_text(prompt):
  result = await genai_client.aio.models.generate_content(
                model='gemini-2.0-flash',
                contents=[
                    types.Part.from_text(prompt),
                    #types.Part.from_uri(video_uri, 'video/mp4')
                ]
            )
  return result.candidates[0].content.parts[0].text

### Get Products Without Descriptions

In [ ]:
sql = """SELECT id, name, brand, category, department, retail_price, sku FROM products WHERE product_description IS NULL;"""
products_df = await run_query(ecom_db_pool, sql)
products_df

### Generate Prompts

In [ ]:
products_df['prompt'] = products_df.apply(
    lambda row: build_product_description_prompt(
        row['name'],
        row['brand'],
        row['category'],
        row['department'],
        row['retail_price'],
        row['sku']
    ),
    axis=1
)
products_df

### Generate Descriptions

> NOTE: You may want to grab some coffee or tea. This step will take about 50 minutes to complete. You can adjust `work_queue` and `num_consumers` to balance processing speed vs throttling/quota limits.

In [ ]:
import asyncio

async def load_queue_from_dataframe(df: pd.DataFrame, queue: asyncio.Queue, num_consumers: int):
    """
    Iterates through DataFrame rows and puts them into the asyncio queue.

    Args:
        df: The Pandas DataFrame to process.
        queue: The asyncio.Queue to put items into.
    """
    logging.info(f"Producer: Starting to load {len(df)} items into the queue...")
    # Use itertuples for efficiency. index=False avoids adding the DataFrame index.
    # name=None uses default namedtuple name 'Pandas'
    for row_tuple in df.itertuples(index=False, name='Product'):
        # Convert the named tuple to a dictionary - often easier for consumers
        item = row_tuple._asdict()
        await queue.put(item)
        logging.info(f"Producer: Put item {item.get('sku', 'N/A')} into queue. Queue size: {queue.qsize()}")
    logging.info("Producer: Finished loading all items.")

    # (Optional but Recommended) Add sentinel values (e.g., None) to signal completion
    # If you have multiple consumers, add one sentinel per consumer.
    # Add one sentinel per consumer
    for _ in range(num_consumers):
        await queue.put(None)
    print(f"Producer: Added {num_consumers} sentinel(s) to queue.")


async def process_items_from_queue(queue: asyncio.Queue, worker_id: int):
    """
    Continuously gets items from the queue and processes them until a sentinel is received.
    """
    logging.info(f"Consumer {worker_id}: Started...")
    while True:
        item = await queue.get()
        if item is None:
            # Sentinel received, signal task completion
            logging.info(f"Consumer {worker_id}: Sentinel received. Exiting.")
            queue.task_done()
            # Put the sentinel back if there are other consumers (not needed here as we only add one)
            # await queue.put(None)
            break # Exit the loop

        # --- Process the item ---
        product_description = await generate_text(item.get('prompt'))
        product_description = product_description.replace("'","''")
        sql = f"UPDATE products SET product_description = '{product_description}' WHERE id = {item.get('id')};"
        await run_query(ecom_db_pool, sql)
        logging.info(f"Consumer {worker_id}: Finished processing ID: {item.get('id')}, SKU: {item.get('sku')}")
        # --- End processing ---

        queue.task_done() # Signal that this item processing is complete


async def generate_product_descriptions_concurrently(products_df):
    # Create the queue. You can optionally set a maxsize.
    # If maxsize is reached, the producer's `await queue.put(item)` will pause
    # until a consumer calls `queue.get()`, providing backpressure.
    work_queue = asyncio.Queue(maxsize=100)
    num_consumers = 10

    # Create tasks for the producer and consumer(s)
    producer_task = asyncio.create_task(load_queue_from_dataframe(products_df, work_queue, num_consumers))

    # Create one or more consumer tasks
    consumer_tasks = []
    for i in range(num_consumers):
        consumer_tasks.append(
            asyncio.create_task(process_items_from_queue(work_queue, i + 1))
        )

    # Wait for the producer to finish loading (optional, but good to ensure all items are queued)
    await producer_task

    # Wait for all consumers to finish processing all items
    # This relies on consumers calling queue.task_done() for each item + the sentinel
    await work_queue.join() # Wait until the queue is fully processed
    logging.info("Main: Queue has been fully processed.")


await generate_product_descriptions_concurrently(products_df)


## Generate Product Images with Imagen 3

> IMPORTANT: This section uses the `imagen-3.0-fast-generate-001` model to generate product images for a dataset containing 29,120 products. As of the time of publishing this notebook, the model costs $0.02 per image. There is a default limit of 1000 images set below to prevent inadvertently running a cost job, but you can adjust that limit up or down as desired.

> NOTE: If you would like to generate pictures of people, ensure your project is allow-listed first. You can request to be allow-listed using [this form](https://docs.google.com/forms/d/e/1FAIpQLSduBp9w84qgim6vLriQ9p7sdz62bMJaL-nNmIVoyiOwd84SMw/viewform).

### Add product_image_uri Column

In [ ]:
sql = """ALTER TABLE products ADD COLUMN product_image_uri TEXT;"""
await run_query(ecom_db_pool, sql)

### build_product_image_prompt()

In [ ]:
def build_product_image_prompt(name, brand):
  prompt = name.replace("'", "")

  # Remove brand and other terms that trigger image generation failures
  remove = [
      brand,
      'Assn',
      '-',
      'Boys',
      'Boy',
      'Girls',
      'Girl',
      'Juniors',
      'Junior',
      '.',
      '&',
      '\n',
  ]
  for r in remove:
      prompt = prompt.lower().replace(r.lower(), '')
  prompt = prompt.strip()
  if not prompt:
    prompt = 'Coming Soon'
  return f"Product image: {prompt}"

### sync_generate_text()  

In [ ]:
def sync_generate_text(prompt):
    result = genai_client.models.generate_content(
                  model='gemini-2.0-flash',
                  contents=[
                      types.Part.from_text(prompt),
                      #types.Part.from_uri(video_uri, 'video/mp4')
                  ]
              )
    return result.candidates[0].content.parts[0].text


### generate_image()

In [ ]:
# Reference: https://cloud.google.com/vertex-ai/generative-ai/docs/image/generate-images
#            https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/imagen-api

import os
import vertexai
from google.cloud import storage
from vertexai.preview.vision_models import ImageGenerationModel

vertexai.init(project=project_id, location=region)
storage_client = storage.Client()
bucket = storage_client.bucket(image_bucket)

model = ImageGenerationModel.from_pretrained("imagen-3.0-fast-generate-001")

@retry(wait=wait_exponential(multiplier=1, min=1, max=10), stop=stop_after_attempt(2), retry=retry_if_exception(retry_condition), before_sleep=before_sleep_log(logging.getLogger(), logging.INFO))
def generate_image(prompt, product_sku, id):

    image_name = f"{product_sku}.png"
    destination_blob_name = f"product-images/{image_name}"
    logging.info(f"Generating image for: {image_name})")

    images = model.generate_images(
        prompt=prompt,
        number_of_images=1,
        language="en",
        add_watermark=True,
        aspect_ratio="1:1",
        safety_filter_level="block_some",
        person_generation="dont_allow",
    )

    if not images.images:
      logging.info(f"RETRY 1: Retrying with a different prompt for {image_name}.")
      rewritten_prompt = sync_generate_text(f"Responding in 1 sentence, simplify this prompt for Imagen3: {prompt}")
      logging.info(f"Modified prompt for id {id} {image_name}: {rewritten_prompt}")

      images = model.generate_images(
          prompt=rewritten_prompt,
          number_of_images=1,
          language="en",
          add_watermark=True,
          aspect_ratio="1:1",
          safety_filter_level="block_only_high",
          person_generation="dont_allow",
      )

      if not images.images:
        logging.warning(f"FAILED: Image generation failed for {image_name}. Prompt: {prompt}")
        return None

    #logging.info(f"Done generating image for: {image_name})")

    # Write the image locally
    #logging.info(f"Writing image locally: {image_name})")
    local_filename = f"{image_name}"
    images[0].save(location=local_filename, include_generation_parameters=False)

    # Upload the image to GCS
    #logging.info(f"Uploading to GCS: {image_name})")
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(local_filename, content_type='image/png')


    # Clean up the local file
    #logging.info(f"Removing the local file: {image_name})")
    os.remove(local_filename)

    #logging.info(f"Returning URI: {image_name})")
    return f"gs://{image_bucket}/{destination_blob_name}" # Return GCS URI

### Get Products Without Product Images

> IMPORTANT: This is the cell that builds the dataset that will be used to generate photos. You can adjust the limit up or down as desired to control the cost of the image generation job.

In [ ]:
# Set the number of products to generate images for here
image_limit = 1000

# Get products without images
sql = f"""SELECT id,
    name,
    brand,
    category,
    department,
    retail_price,
    sku,
    product_description
  FROM products
  WHERE product_image_uri IS NULL
  AND name IS NOT NULL
  AND brand IS NOT NULL
  LIMIT {image_limit};"""
products_df = await run_query(ecom_db_pool, sql)
products_df

### Build Image Prompts

In [ ]:
products_df['image_prompt'] = products_df.apply(
    lambda row: build_product_image_prompt(
        row['name'],
        row['brand'],
    ),
    axis=1
)
products_df

### Generate Images

This step will take 4-5 hours to generate images for all 29,120 products in the Cymbal Shops product catalog, running 150 async requests at a time. You can adjust the number of images to generate in the cells above (see comments). You can also request a quota increase to allow more concurrent invocations, in which case you can increase the `num_consumers` variable below.

> NOTE: Errors are expected in this step due to ambiguous product names and content filter false positives. Failures will be marked in the `product_image_uri` column in the database. Early testing resulted in an ~80% success rate. You can tweak the prompt and retry for failed items if desired.

In [ ]:
import asyncio


async def load_queue_from_dataframe(df: pd.DataFrame, queue: asyncio.Queue, num_consumers: int):
    """
    Iterates through DataFrame rows and puts them into the asyncio queue.

    Args:
        df: The Pandas DataFrame to process.
        queue: The asyncio.Queue to put items into.
    """
    print(f"Producer: Starting to load {len(df)} items into the queue...")
    # Use itertuples for efficiency. index=False avoids adding the DataFrame index.
    # name=None uses default namedtuple name 'Pandas'
    for row_tuple in df.itertuples(index=False, name='Product'):
        # Convert the named tuple to a dictionary - often easier for consumers
        item = row_tuple._asdict()
        await queue.put(item)
        logging.info(f"Producer: Put item {item.get('sku', 'N/A')} into queue. Queue size: {queue.qsize()}")
    logging.info("Producer: Finished loading all items.")

    # Add one sentinel value (e.g., None) per consumer to signal completion
    for _ in range(num_consumers):
        await queue.put(None)
    print(f"Producer: Added {num_consumers} sentinel(s) to queue.")


async def process_items_from_queue(queue: asyncio.Queue, worker_id: int):
    """
    Continuously gets items from the queue and processes them until a sentinel is received.
    Propagates exceptions if processing fails.
    """
    logging.info(f"Consumer {worker_id}: Started...")
    while True:
        item = await queue.get()

        # --- Check for Sentinel ---
        if item is None:
            print(f"Consumer {worker_id}: Sentinel received. Exiting.")
            queue.task_done() # Mark sentinel processing as done
            break # Exit the loop

        # --- Process the item ---

        try:
            log_prefix = f"Consumer {worker_id}: Item ID {item.get('id', 'N/A')}:"
            logging.info(f"{log_prefix} Starting processing.")

            # Get current running loop
            loop = asyncio.get_running_loop()

            # Run the blocking image generation function
            # Ensure generate_image raises exceptions on failure or returns None clearly
            image_uri = await loop.run_in_executor(
                None, # Use default executor (ThreadPoolExecutor)

                # --- Define the blocking function to run ---
                generate_image,

                # --- Add function parameters here ---
                item.get('image_prompt'), # First argument for generate_image
                item.get('sku'),          # Second argument for generate_image, etc
                item.get('id'),
            )

            # Handle image generation failure (if it returns None instead of raising)
            if image_uri is None:
                # Log warning and continue (skip DB update for this item)
                logging.warning(f"{log_prefix} Image generation failed or returned None.")
                sql = f"UPDATE products SET product_image_uri = 'FAILED - Prompt: {item.get('image_prompt')}' WHERE id = {item.get('id')};"
                await run_query(ecom_db_pool, sql)
            else:
                logging.info(f"{log_prefix} Image generated: '{image_uri}'")
                # Run the database update
                sql = f"UPDATE products SET product_image_uri = '{image_uri}' WHERE id = {item.get('id')};"
                await run_query(ecom_db_pool, sql)
                #logging.info(f"{log_prefix} Database updated successfully.")

            # --- Processing successful for this item ---
            queue.task_done() # Signal completion ONLY on success

        except Exception as e:
            # Log the exception WITH traceback
            logging.error(f"Consumer {worker_id}: Unhandled exception processing item ID {item.get('id', 'N/A')}: {e}", exc_info=True)
            # Log error without raising so that remaining items can be processed.
            logging.warning(e)


async def process_dataframe_concurrently(products_df):
    # Create the queue with maxsize. If maxsize is reached, the producer's
    # `await queue.put(item)` will pause until a consumer calls `queue.get()`, providing backpressure.

    # --- Set queue max size and number of consumers/workers here ---
    work_queue = asyncio.Queue(maxsize=400)
    num_consumers = 150
    all_tasks = []

    # --- Create Tasks ---
    print("Main: Creating producer task...")
    producer_task = asyncio.create_task(
        load_queue_from_dataframe(products_df, work_queue, num_consumers),
        name="Producer"
    )
    all_tasks.append(producer_task)

    print(f"Main: Creating {num_consumers} consumer tasks...")
    for i in range(num_consumers):
        consumer_task = asyncio.create_task(
            process_items_from_queue(work_queue, i + 1),
            name=f"Consumer-{i+1}"
        )
        all_tasks.append(consumer_task)

    # Wait for the producer to finish loading (optional, but good to ensure all items are queued)
    await producer_task

    # --- Run Tasks and Handle Completion/Failure ---
    done, pending = [], []
    try:
        # Wait for all tasks to complete. gather will raise the *first* exception
        # encountered in any of the tasks.
        print("Main: Waiting for tasks to complete...")
        # Use asyncio.wait instead of gather to have more control over pending tasks on error
        done, pending = await asyncio.wait(all_tasks, return_when=asyncio.FIRST_COMPLETED)

        # Check if any completed tasks raised an exception
        for task in done:
            if task.exception():
                raise task.exception() # Raise the exception from the failed task

    except Exception as e:
        print(f"Main: An error occurred in a task: {e}", exc_info=True)
        print("Main: Attempting to cancel pending tasks...")
        for task in pending: # Cancel tasks found in the pending set from asyncio.wait
             task.cancel()

        # Give cancelled tasks a moment to process the cancellation
        # and gather any CancelledError exceptions (optional but cleaner)
        if pending:
             await asyncio.wait(pending, timeout=1.0) # Wait briefly

        # Important: Re-raise the original exception to stop the program execution
        # Or handle it appropriately (e.g., sys.exit(1))
        raise e # Propagate the error out

    finally:
        # Ensure all tasks are truly finished one way or another (optional cleanup)
        remaining_tasks = [t for t in all_tasks if not t.done()]
        if remaining_tasks:
             print("Main: Waiting for final cleanup of any remaining tasks...")
             await asyncio.wait(remaining_tasks, timeout=1.0) # Brief wait

    # If execution reaches here, it means all tasks finished without unhandled exceptions propagating
    print("Main: Process finished.")

# This is a very verbose process. Changing the logging level to WARNING
logging.basicConfig(level=logging.WARNING, stream=sys.stdout, format='%(asctime)s[%(levelname)5s][%(name)14s] - %(message)s',  datefmt='%H:%M:%S', force=True)

# Kick off parallel image creation
await process_dataframe_concurrently(products_df)

# Switch logging back to INFO
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(asctime)s[%(levelname)5s][%(name)14s] - %(message)s',  datefmt='%H:%M:%S', force=True)


### View Image Generation Success Rate

In [ ]:
sql = """WITH a AS (
  SELECT
    (SELECT COUNT(*)  FROM products WHERE product_image_uri IS NOT NULL) AS processed,
    (SELECT COUNT(*) FROM products WHERE product_image_uri LIKE 'FAILED%') AS failed
) SELECT processed,
  processed - failed AS successful,
  failed,
  1-(failed/processed::FLOAT) AS success_rate
  FROM a;"""
await run_query(ecom_db_pool, sql)

## Generate Multimodal Embeddings

### Add Embedding Columns

In [ ]:
sql_array = []
sql_array.append("ALTER TABLE products ADD COLUMN product_description_embedding VECTOR(1408);")
sql_array.append("ALTER TABLE products ADD COLUMN product_description_embedding_model TEXT;")
sql_array.append("ALTER TABLE products ADD COLUMN product_image_embedding VECTOR(1408);")
sql_array.append("ALTER TABLE products ADD COLUMN product_image_embedding_model TEXT;")
for sql in sql_array:
  await run_query(ecom_db_pool, sql)

### generate_multimodal_embeddings()

In [ ]:
import vertexai
from vertexai.vision_models import Image, MultiModalEmbeddingModel

vertexai.init(project=project_id, location=region)
model = MultiModalEmbeddingModel.from_pretrained("multimodalembedding@001")

@retry(wait=wait_fixed(10), stop=stop_after_attempt(7), retry=retry_if_exception(retry_condition))
def generate_multimodal_embeddings(uri, text):

  image = Image.load_from_file(f"{uri}")

  embeddings = model.get_embeddings(
      image=image,
      contextual_text=text,
      dimension=1408,
  )
  return embeddings

### Get Products to Embed

In [ ]:
sql = """SELECT id,
            name,
            brand,
            category,
            department,
            retail_price,
            sku,
            product_description,
            product_image_uri
         FROM products
         WHERE product_image_uri IS NOT NULL
            AND product_image_uri NOT LIKE 'FAILED%'
            AND product_image_embedding IS NULL
         LIMIT 30000;"""
products_df = await run_query(ecom_db_pool, sql)
products_df

### Generate Embeddings

In [ ]:
import asyncio
import vertexai
from vertexai.vision_models import Image, MultiModalEmbeddingModel


async def load_queue_from_dataframe(df: pd.DataFrame, queue: asyncio.Queue, num_consumers: int):
    """
    Iterates through DataFrame rows and puts them into the asyncio queue.

    Args:
        df: The Pandas DataFrame to process.
        queue: The asyncio.Queue to put items into.
    """
    print(f"Producer: Starting to load {len(df)} items into the queue...")
    # Use itertuples for efficiency. index=False avoids adding the DataFrame index.
    # name=None uses default namedtuple name 'Pandas'
    for row_tuple in df.itertuples(index=False, name='Product'):
        # Convert the named tuple to a dictionary - often easier for consumers
        item = row_tuple._asdict()
        await queue.put(item)
        logging.info(f"Producer: Put item {item.get('sku', 'N/A')} into queue. Queue size: {queue.qsize()}")
    logging.info("Producer: Finished loading all items.")

    # Add one sentinel value (e.g., None) per consumer to signal completion
    for _ in range(num_consumers):
        await queue.put(None)
    print(f"Producer: Added {num_consumers} sentinel(s) to queue.")


async def process_items_from_queue(queue: asyncio.Queue, worker_id: int):
    """
    Continuously gets items from the queue and processes them until a sentinel is received.
    Propagates exceptions if processing fails.
    """
    logging.info(f"Consumer {worker_id}: Started...")
    while True:
        item = await queue.get()

        # --- Check for Sentinel ---
        if item is None:
            print(f"Consumer {worker_id}: Sentinel received. Exiting.")
            queue.task_done() # Mark sentinel processing as done
            break # Exit the loop

        # --- Process the item ---

        try:
            log_prefix = f"Consumer {worker_id}: Item ID {item.get('id', 'N/A')}:"
            logging.info(f"{log_prefix} Starting processing.")

            # Get current running loop
            loop = asyncio.get_running_loop()

            # Define variables
            prompt = f"Product Name: {item.get('name')}\n\nProduct Description: {item.get('product_description')}"
            prompt = prompt.replace("'","")
            uri = item.get('product_image_uri')

            # Run the blocking embedding generation function
            result = await loop.run_in_executor(
                None, # Use default executor (ThreadPoolExecutor)

                # --- Define the blocking function to run ---
                generate_multimodal_embeddings,

                # --- Add function parameters here ---
                uri,      # First argument for generate_multimodal_embeddings
                prompt,   # Second argument for generate_multimodal_embeddings, etc
            )

            # Handle embedding generation failure (if it returns None instead of raising)
            if result is None:
                # Log warning and continue (skip DB update for this item)
                logging.warning(f"{log_prefix} Embedding generation failed or returned None.")
            else:
                # Run the database update
                sql = f"""UPDATE products
                    SET product_image_embedding = '{result.image_embedding}',
                        product_image_embedding_model = 'multimodalembedding@001',
                        product_description_embedding = '{result.text_embedding}',
                        product_description_embedding_model = 'multimodalembedding@001'
                    WHERE id = {item.get('id')};"""
                await run_query(ecom_db_pool, sql)

            # --- Processing successful for this item ---
            queue.task_done() # Signal completion ONLY on success

        except Exception as e:
            # Log the exception WITH traceback
            logging.error(f"Consumer {worker_id}: Unhandled exception processing item ID {item.get('id', 'N/A')}: {e}", exc_info=True)
            # Log error without raising so that remaining items can be processed.
            logging.warning(e)


async def process_dataframe_concurrently(products_df):
    # Create the queue with maxsize. If maxsize is reached, the producer's
    # `await queue.put(item)` will pause until a consumer calls `queue.get()`, providing backpressure.

    # --- Set queue max size and number of consumers/workers here ---
    work_queue = asyncio.Queue(maxsize=800)
    num_consumers = 100
    all_tasks = []

    # --- Create Tasks ---
    print("Main: Creating producer task...")
    producer_task = asyncio.create_task(
        load_queue_from_dataframe(products_df, work_queue, num_consumers),
        name="Producer"
    )
    all_tasks.append(producer_task)

    print(f"Main: Creating {num_consumers} consumer tasks...")
    for i in range(num_consumers):
        consumer_task = asyncio.create_task(
            process_items_from_queue(work_queue, i + 1),
            name=f"Consumer-{i+1}"
        )
        all_tasks.append(consumer_task)

    # Wait for the producer to finish loading (optional, but good to ensure all items are queued)
    await producer_task

    # --- Run Tasks and Handle Completion/Failure ---
    done, pending = [], []
    try:
        # Wait for all tasks to complete. gather will raise the *first* exception
        # encountered in any of the tasks.
        print("Main: Waiting for tasks to complete...")
        # Use asyncio.wait instead of gather to have more control over pending tasks on error
        done, pending = await asyncio.wait(all_tasks, return_when=asyncio.FIRST_COMPLETED)

        # Check if any completed tasks raised an exception
        for task in done:
            if task.exception():
                raise task.exception() # Raise the exception from the failed task

    except Exception as e:
        print(f"Main: An error occurred in a task: {e}", exc_info=True)
        print("Main: Attempting to cancel pending tasks...")
        for task in pending: # Cancel tasks found in the pending set from asyncio.wait
             task.cancel()

        # Give cancelled tasks a moment to process the cancellation
        # and gather any CancelledError exceptions (optional but cleaner)
        if pending:
             await asyncio.wait(pending, timeout=1.0) # Wait briefly

        # Important: Re-raise the original exception to stop the program execution
        # Or handle it appropriately (e.g., sys.exit(1))
        raise e # Propagate the error out

    finally:
        # Ensure all tasks are truly finished one way or another (optional cleanup)
        remaining_tasks = [t for t in all_tasks if not t.done()]
        if remaining_tasks:
             print("Main: Waiting for final cleanup of any remaining tasks...")
             await asyncio.wait(remaining_tasks, timeout=1.0) # Brief wait

    # If execution reaches here, it means all tasks finished without unhandled exceptions propagating
    print("Main: Process finished.")

# This is a very verbose process. Changing the logging level to WARNING
logging.basicConfig(level=logging.WARNING, stream=sys.stdout, format='%(asctime)s[%(levelname)5s][%(name)14s] - %(message)s',  datefmt='%H:%M:%S', force=True)

# Kick off parallel image creation
await process_dataframe_concurrently(products_df)

# Switch logging back to INFO
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(asctime)s[%(levelname)5s][%(name)14s] - %(message)s',  datefmt='%H:%M:%S', force=True)



### View Embedding Generation Success Rate

In [ ]:
sql = """WITH a AS (
  SELECT
    (SELECT COUNT(*) FROM products WHERE product_image_embedding IS NOT NULL) AS total_products_with_image_embeddings,
    (SELECT COUNT(*) FROM products WHERE product_image_uri IS NOT NULL AND product_image_uri NOT LIKE 'FAILED%') AS total_products_with_image,
    (SELECT COUNT(*) FROM products) AS total_products
) SELECT
  total_products,
  total_products_with_image,
  total_products_with_image/total_products::float AS percent_with_image,
  total_products_with_image_embeddings,
  total_products_with_image_embeddings/total_products::float AS percent_with_embeddings
  FROM a;"""

await run_query(ecom_db_pool, sql)

## Generate BM25 Sparse Embeddings

### Create or Instantiate Bucket for BM25 Index Data

In [ ]:
# Create index_bucket if none is provided
import uuid
from google.cloud import storage

storage_client = storage.Client()
random_suffix = uuid.uuid4().hex[:6]  # Get a 6-character hexadecimal suffix
index_bucket_name = f"bm25-index-{random_suffix}"

if not index_bucket:
    index_bucket = storage_client.create_bucket(index_bucket_name)
    print(f"Bucket {index_bucket.name} created.")
else:
    print(f"Using provided index_bucket: {index_bucket}")
    index_bucket = storage_client.bucket(index_bucket)

### Build BM25 Index from AlloyDB Content

In [ ]:
from pymilvus.model.sparse import BM25EmbeddingFunction

# Switch logging back to INFO
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(asctime)s[%(levelname)5s][%(name)14s] - %(message)s',  datefmt='%H:%M:%S', force=True)


# Instantiate BM25 model
bm25_ef = BM25EmbeddingFunction()

# Get text content from the vector store
sql = "SELECT id, 'Name: ' || COALESCE(name,'') || '\n Category: ' || COALESCE(category,'') || '\n Brand: ' || COALESCE(brand,'') || '\n Department: ' || COALESCE(department,'') AS content FROM products;"
docs = await run_query(ecom_db_pool, sql)
docs = docs.replace("'","")

# Fit the model to the AlloyDB content
bm25_ef.fit(docs['content'].to_list())


### Upload Model Parameters to GCS

In [ ]:
import os

# Store the fitted parameters to expedite future processing.
bm25_params_file_name = "bm25_params.json"
bm25_ef.save(bm25_params_file_name)

# Upload saved model parameters to GCS
current_directory = os.getcwd()
blob_path = f"bm25_index/{bm25_params_file_name}"
file_name = f"{current_directory}/{bm25_params_file_name}"
blob = index_bucket.blob(blob_path)
blob.upload_from_filename(file_name)
print(f"Uploaded model parameters from: {file_name} to gs://{index_bucket.name}/{blob_path}")

### Download Model Parameters from GCS

In [ ]:
# Download model parameters (update blob to use your own file)
blob = index_bucket.blob(f"bm25_index/{bm25_params_file_name}")
blob.download_to_filename(bm25_params_file_name)

# Load the saved params (optionally provide your own index.json)
bm25_ef = BM25EmbeddingFunction()
bm25_ef.load(bm25_params_file_name)

# Print out the max dims:
max_1_based_dims = bm25_ef.dim + 1
print(f"Max dimensionality: {max_1_based_dims}")

### Define Helper Function for `sparsevec` Encoding

In [ ]:
def encode_sparsevec(query: str, dimensions: int = max_1_based_dims) -> str:

    # Generate the sparse embeddings
    sparse_embeddings = bm25_ef.encode_queries([query])
    lil = sparse_embeddings.tolil(copy=False)
    sparse_scores, sparse_indices = lil.data.tolist()[0], lil.rows.tolist()[0]

    # Ensure sparse_scores and sparse_indices lists are the same length
    assert len(sparse_scores) == len(sparse_indices)

    # sparsevec data type is 1-based. sparse_indices are zero-based.
    sparse_indices = [x + 1 for x in sparse_indices]

    # Zip results and transform to expected format for pgvector sparsevec type
    result = [f"{key}:{value:.7g}" for key, value in zip(sparse_indices, sparse_scores)]
    return f"{{{','.join(result)}}}/{dimensions}"

### Create Sparse Embeddings for Content

In [ ]:
docs['sparse_embedding'] = docs['content'].apply(encode_sparsevec)
docs

### Add `sparsevec` Column to AlloyDB Vector Store

In [ ]:
sql = f"ALTER TABLE products ADD COLUMN sparse_embedding sparsevec({bm25_ef.dim + 1})"
await run_query(ecom_db_pool, sql)

### Update AlloyDB with Sparse Embeddings

In [ ]:
for row in docs.itertuples():
  sql = f"""
  UPDATE products SET sparse_embedding = '{row.sparse_embedding}'
  WHERE id = '{row.id}'
  """
  #print(sql)
  await run_query(ecom_db_pool, sql)